# 06b — QAT Inference (ModelOpt INT8 / INT4)

Loads ModelOpt QAT-trained checkpoints and measures inference time + accuracy on the ImageNet-100 subset.

| Precision | Config used during training | Weights | Activations |
|---|---|---|---|
| `int8` | `INT8_DEFAULT_CFG` | per-channel INT8 | per-tensor INT8 |
| `int4` | `INT4_BLOCKWISE_WEIGHT_ONLY_CFG` | blockwise INT4 | fp (unchanged) |

Each result is saved to `runs/` using the same JSON schema as PTQ notebooks.

In [ ]:
from pathlib import Path
import sys, os, json, time

PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(SRC / "qat_modelopt") not in sys.path:
    sys.path.insert(0, str(SRC / "qat_modelopt"))

In [ ]:
import torch
import torch.nn as nn
import pandas as pd

import modelopt.torch.opt as mto

from config import ExperimentConfig, set_seed
from data import build_runner_loaders
from eval import evaluate
from utils import load_runs, flatten_runs
from qat_modelopt.quantize import get_model, get_quant_cfg, quantize_model

pd.set_option("display.max_columns", 200)

## 1. Configuration

In [ ]:
CHECKPOINT_ROOT  = PROJECT_ROOT / "checkpoints"
RUNS_ROOT        = PROJECT_ROOT / "runs"

NUM_EVAL_BATCHES = 500
BATCH_SIZE       = 1
SEED             = 42
NUM_CLASSES      = 100

# ---- ModelOpt QAT runs to evaluate ----
# Each entry: (model_precision, input_quant_bits, checkpoint_subdir)
# checkpoint_subdir is relative to CHECKPOINT_ROOT.
# Add new rows here as more checkpoints become available.
QAT_RUNS = [
    ("int8", 8, "qat_modelopt/resnet18_modelopt_int8_in8b_cuda_bs256"),
    ("int8", 4, "qat_modelopt/resnet18_modelopt_int8_in4b_cuda_bs256"),
    ("int4", 8, "qat_modelopt/resnet18_modelopt_int4_in8b_cuda_bs256"),
    # ("int4", 4, "qat_modelopt/resnet18_modelopt_int4_in4b_cuda_bs256"),
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

## 2. Helper — load one ModelOpt QAT checkpoint

ModelOpt checkpoints come in two files:
- `qat_modelopt_best.pth` — standard model weights
- `qat_modelopt_best_mostate.pt` — quantizer scales / zero-points

The mostate must be restored **before** loading the weight state_dict so that
the quantized layer parameters already exist in the model.

In [ ]:
def load_qat_modelopt_checkpoint(
    model_precision: str,
    checkpoint_subdir: str,
) -> nn.Module:
    """
    Reconstruct a ModelOpt-quantized model from its two checkpoint files.
    Returns an eval-mode model on `device`.
    """
    ckpt_dir  = CHECKPOINT_ROOT / checkpoint_subdir
    ckpt_path = ckpt_dir / "qat_modelopt_best.pth"
    mo_path   = ckpt_dir / "qat_modelopt_best_mostate.pt"

    # 1. Load plain FP32 architecture
    model = get_model(str(CHECKPOINT_ROOT / "best.pth"), num_classes=NUM_CLASSES)
    model = model.to(device)

    # 2. Restore quantizer structure (inserts fake-quant nodes matching training config)
    print(f"  [mostate] {mo_path}")
    mo_state = torch.load(str(mo_path), map_location="cpu")
    mto.restore_from_modelopt_state(model, mo_state)

    # 3. Load fine-tuned weights
    print(f"  [weights] {ckpt_path}")
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    model.load_state_dict(ckpt["model"])

    return model.eval()

## 3. Helper — run one QAT evaluation

In [ ]:
def run_qat_inference(
    model_precision: str,
    input_quant_bits: int,
    checkpoint_subdir: str,
) -> dict:
    """
    Load a ModelOpt QAT checkpoint, evaluate on ImageNet-100 val, and return
    a result payload matching the PTQ JSON schema.
    """
    run_id = (
        f"resnet18_qat_modelopt_{model_precision}"
        f"_in{input_quant_bits}b_{device.type}_bs{BATCH_SIZE}"
    )

    print(f"\n{'='*70}")
    print(f"[QAT-ModelOpt] {run_id}")
    print(f"{'='*70}")

    cfg = ExperimentConfig(
        backend="pytorch",
        device=device.type,
        batch_size=BATCH_SIZE,
        seed=SEED,
        num_eval_batches=NUM_EVAL_BATCHES,
        input_quant_bits=input_quant_bits,
        model_precision="fp32",   # fake-quant model runs in fp32
        num_classes=NUM_CLASSES,
    )
    set_seed(cfg)

    model  = load_qat_modelopt_checkpoint(model_precision, checkpoint_subdir)
    _, loader = build_runner_loaders(cfg)

    criterion = nn.CrossEntropyLoss()
    t0      = time.perf_counter()
    tracker = evaluate(model, loader, cfg, criterion=criterion)
    total_eval_time = round(time.perf_counter() - t0, 3)

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": {
            **cfg.to_dict(),
            "backend": "qat_modelopt",
            "model_precision": model_precision,
            "qat_checkpoint": str(CHECKPOINT_ROOT / checkpoint_subdir),
        },
        "results": tracker.summary(),
        "error": None,
        "total_eval_time_sec": total_eval_time,
    }
    return payload

## 4. Run all QAT evaluations & save results

In [ ]:
records = []

for precision, in_bits, ckpt_dir in QAT_RUNS:
    payload = run_qat_inference(precision, in_bits, ckpt_dir)
    records.append(payload)

    out_dir = RUNS_ROOT / payload["run_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    result_path = out_dir / "result.json"
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    print(f"[Saved] {result_path}")

for r in records:
    print(f"{r['run_id']}  status={r['status']}  top1={r['results'].get('top1_acc', 'N/A')}")

## 5. Summary table

In [ ]:
runs = load_runs(str(RUNS_ROOT), status="ok")
rows = flatten_runs(runs)
df   = pd.DataFrame(rows)

df_qat = df[df["cfg.backend"] == "qat_modelopt"].copy()

df_qat[[
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.loss_avg",
    "res.infer_ms_avg",
    "res.infer_ms_std",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.input_quant_bits"])

## 6. ModelOpt QAT vs pytorch-quantization QAT comparison

In [ ]:
df_qat_pq = df[df["cfg.backend"] == "qat_pytorch"].copy()   # pytorch-quantization results

cols = ["run_id", "cfg.model_precision", "cfg.input_quant_bits",
        "res.top1_acc", "res.top5_acc", "res.infer_ms_avg"]

compare = pd.concat([
    df_qat[cols],
    df_qat_pq[cols],
]).sort_values(["cfg.input_quant_bits", "cfg.model_precision", "run_id"])

compare

## 7. ModelOpt QAT vs FP32 baseline comparison

In [ ]:
df_fp32 = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.model_precision"] == "fp32")
    & (df["cfg.device"] == "cuda")
].copy()

cols = ["run_id", "cfg.input_quant_bits", "res.top1_acc", "res.top5_acc", "res.infer_ms_avg"]

compare2 = pd.concat([
    df_qat[cols],
    df_fp32[cols],
]).sort_values(["cfg.input_quant_bits", "run_id"])

compare2